# ml_E_unsup_codes.ipynb

**所属章节**：Chapter E 无监督学习  
**用途**：生成讲义配图（7张），保存至 `figs/` 目录  
**运行说明**：顺序执行，约 1–2 分钟

**输出文件**：
- `figs/fig_E_pca_2d.png`：PCA 几何示意
- `figs/fig_E_scree_plot.png`：碎石图
- `figs/fig_E_kmeans_steps.png`：K-means 迭代过程
- `figs/fig_E_elbow.png`：肘部法则图
- `figs/fig_E_dendrogram.png`：层次聚类树状图
- `figs/fig_E_silhouette.png`：轮廓系数图
- `figs/fig_E_isolation_forest.png`：孤立森林异常检测


In [ ]:
# Global setup
import os
os.environ.setdefault('MPLCONFIGDIR', os.path.join(os.getcwd(), '.mplconfig'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.ensemble import IsolationForest
from sklearn.datasets import make_blobs, make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)

# Chinese font support
from matplotlib import font_manager

candidate_fonts = [
    'Microsoft YaHei', 'SimHei', 'SimSun', 'Arial Unicode MS',
    'Noto Sans CJK SC', 'Source Han Sans SC', 'WenQuanYi Micro Hei'
]
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
zh_fonts = [f for f in candidate_fonts if f in available_fonts]
ZH_FONT = zh_fonts[0] if zh_fonts else None
if zh_fonts:
    plt.rcParams['font.sans-serif'] = zh_fonts + plt.rcParams['font.sans-serif']
else:
    print('Warning: no common Chinese font found; Chinese text may render as boxes.')
FP = font_manager.FontProperties(family=ZH_FONT) if ZH_FONT else font_manager.FontProperties()
FPB = font_manager.FontProperties(family=ZH_FONT, weight='bold') if ZH_FONT else font_manager.FontProperties(weight='bold')
plt.rcParams['axes.unicode_minus'] = False

plt.rcParams.update({'figure.dpi':120, 'savefig.dpi':300,
    'font.size':11,'axes.titlesize':13,'axes.labelsize':11,
    'xtick.labelsize':10,'ytick.labelsize':10,'legend.fontsize':10,
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.25, 'grid.linestyle':'--'})
C = {'primary':'#0B3D91','secondary':'#B8860B','tertiary':'#2F5E9E',
     'neutral':'#878787','highlight':'#6B4E00','fill':'#D6E2F3'}
SEED=42; np.random.seed(SEED)
print('Global setup complete')


---
## 图1：PCA 几何示意（2D）


In [ ]:
# 生成二维相关数据
cov = [[1.0, 0.8],[0.8, 1.0]]
X2d = np.random.multivariate_normal([0,0], cov, 200)

pca2 = PCA(n_components=2)
pca2.fit(X2d)
v1, v2 = pca2.components_[0], pca2.components_[1]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(X2d[:,0], X2d[:,1],
           color=C['primary'], alpha=0.4, s=20, label='观测点')

# 绘制主成分方向（长度按方差缩放）
for i, (v, col, lbl, lw) in enumerate([
    (v1, C['highlight'],  f'PC1 (EVR={pca2.explained_variance_ratio_[0]:.1%})', 2.5),
    (v2, C['secondary'],  f'PC2 (EVR={pca2.explained_variance_ratio_[1]:.1%})', 2.0)
]):
    scale = np.sqrt(pca2.explained_variance_[i]) * 2
    ax.annotate('', xy=(v*scale), xytext=(-v*scale),
                arrowprops=dict(arrowstyle='->', color=col, lw=lw))
    ax.text(v[0]*scale*1.15, v[1]*scale*1.15, lbl,
            color=col, fontsize=10, ha='center')

# 画一个样本点到 PC1 的投影
sample = X2d[10]
proj   = np.dot(sample, v1) * v1
ax.plot([sample[0], proj[0]], [sample[1], proj[1]],
        'k--', lw=1, alpha=0.5)
ax.scatter(*sample, color='k', s=60, zorder=5)
ax.scatter(*proj,   color=C['highlight'], s=60, zorder=5,
           marker='D', label='投影到 PC1')

ax.set_aspect('equal')
ax.set_xlim(-3.5,3.5); ax.set_ylim(-3.5,3.5)
ax.set_xlabel('X1'); ax.set_ylabel('X2')
ax.set_title('PCA 几何示意：主成分方向')
ax.legend(fontsize=10)
ax.axhline(0,color='k',lw=0.5,alpha=0.3)
ax.axvline(0,color='k',lw=0.5,alpha=0.3)
fig.tight_layout()
fig.savefig('figs/fig_E_pca_2d.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_E_pca_2d.png 已保存')

---
## 图2：碎石图（Scree Plot）


In [ ]:
# 生成高维数据（n=300, p=20，真实秩≈5）
n_comp = 5
X_low = np.random.randn(300, n_comp)
A = np.random.randn(n_comp, 20)
X_high = X_low @ A + 0.5 * np.random.randn(300, 20)
X_high_sc = StandardScaler().fit_transform(X_high)

pca_h = PCA(n_components=10)
pca_h.fit(X_high_sc)
evr  = pca_h.explained_variance_ratio_
cevr = evr.cumsum()

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax2 = ax1.twinx()

bars = ax1.bar(range(1,11), evr, color=C['primary'], alpha=0.75, label='各主成分 EVR')
ax2.plot(range(1,11), cevr, 'o-', color=C['secondary'],
         lw=2, ms=6, label='累积 EVR')
ax2.axhline(0.8, color=C['neutral'], ls='--', lw=1.2, label='80% 阈值')

# 标注每个柱的值
for i, v in enumerate(evr):
    ax1.text(i+1, v+0.005, f'{v:.1%}', ha='center', fontsize=10, color=C['primary'])

ax1.set_xlabel('主成分编号')
ax1.set_ylabel('方差解释比（EVR）', color=C['primary'])
ax2.set_ylabel('累积方差解释比', color=C['secondary'])
ax2.set_ylim(0, 1.05)
ax1.set_title('碎石图（Scree Plot）')

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labs1+labs2, fontsize=10, loc='center right')

ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)
fig.tight_layout()
fig.savefig('figs/fig_E_scree_plot.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_E_scree_plot.png 已保存')

---
## 图3：K-means 迭代过程（4步）


In [ ]:
# 三簇数据
X_km, y_km = make_blobs(n_samples=120, centers=3,
                          cluster_std=0.8, random_state=SEED)
colors3 = [C['primary'], C['secondary'], C['tertiary']]

# 手动模拟 K-means 迭代（展示过程）
K = 3
# 固定初始质心（随机选3个点）
np.random.seed(0)
init_idx = np.random.choice(len(X_km), K, replace=False)
centroids = X_km[init_idx].copy()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
fig.subplots_adjust(wspace=0.15)
n_show = 4  # 展示4步

for step in range(n_show):
    ax = axes[step]
    # 分配步骤
    dists   = np.linalg.norm(X_km[:,None] - centroids[None], axis=2)
    labels  = dists.argmin(axis=1)
    # 绘制样本点
    for k in range(K):
        mask = labels == k
        ax.scatter(X_km[mask,0], X_km[mask,1],
                   color=colors3[k], alpha=0.55, s=20)
    # 绘制质心
    for k in range(K):
        ax.scatter(centroids[k,0], centroids[k,1],
                   color=colors3[k], marker='X', s=150,
                   edgecolors='k', lw=1.2, zorder=5)
    ax.set_title(f'第 {step+1} 步')
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines['top'].set_visible(True); ax.spines['right'].set_visible(True)
    # 箭头指向新质心
    if step < n_show - 1:
        new_centroids = np.array([X_km[labels==k].mean(0) for k in range(K)])
        for k in range(K):
            ax.annotate('', xy=new_centroids[k], xytext=centroids[k],
                        arrowprops=dict(arrowstyle='->', color='k',
                                        lw=1.2, alpha=0.6))
        centroids = new_centroids
    else:
        ax.set_title(f'第 {step+1} 步（收敛）')

fig.suptitle('K-means 迭代过程（K=3）：X 为质心，箭头为质心更新方向')
fig.savefig('figs/fig_E_kmeans_steps.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_E_kmeans_steps.png 已保存')

---
## 图4：肘部法则（Elbow Method）


In [ ]:
wcss_list = []
K_range   = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    km.fit(X_km)
    wcss_list.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(K_range, wcss_list, 'o-', color=C['primary'], lw=2.2, ms=7)

# 标注肘点（K=3）
ax.axvline(3, color=C['highlight'], lw=1.5, ls='--')
ax.annotate('肘点（K=3）',
            xy=(3, wcss_list[2]),
            xytext=(5, wcss_list[2]+20),
            fontsize=10, color=C['highlight'],
            arrowprops=dict(arrowstyle='->', color=C['highlight'], lw=1.2))

ax.set_xlabel('聚类数 K')
ax.set_ylabel('WCSS（组内距离平方和）')
ax.set_title('肘部法则：选择最优聚类数 K')
ax.set_xticks(list(K_range))
fig.tight_layout()
fig.savefig('figs/fig_E_elbow.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_E_elbow.png 已保存')

---
## 图5：层次聚类树状图


In [ ]:
# 模拟 A 股行业数据（10 个行业，用收益率相关性聚类）
industries = ['银行','保险','券商','房地产','建筑',
               '钢铁','有色','化工','医药','消费']
n_ind = len(industries)

# 构造相关矩阵（金融/地产相关，周期性行业相关）
np.random.seed(SEED)
base = np.random.randn(200, n_ind)
# 同行业相关性更高
base[:, :3]  += np.random.randn(200,1) * 1.5   # 金融板块
base[:, 3:5] += np.random.randn(200,1) * 1.2   # 地产建筑
base[:, 5:8] += np.random.randn(200,1) * 1.2   # 周期品
base[:, 8:]  += np.random.randn(200,1) * 1.0   # 消费医药
corr_mat = np.corrcoef(base.T)

# 用 1 - 相关系数作为距离
dist = 1 - corr_mat
np.fill_diagonal(dist, 0)

# 层次聚类（Ward 方法）
from scipy.spatial.distance import squareform
condensed = squareform(dist, checks=False)
Z = linkage(condensed, method='ward')

fig, ax = plt.subplots(figsize=(9, 4.5))
dendrogram(Z, labels=industries, ax=ax,
           color_threshold=0.7*max(Z[:,2]),
           above_threshold_color=C['neutral'],
           leaf_font_size=11)
ax.axhline(0.7*max(Z[:,2]), color=C['highlight'], ls='--',
           lw=1.5, label='切割线（5 个簇）')
ax.set_ylabel('合并距离（1 - 相关系数）')
ax.set_title('A 股行业层次聚类树状图（Ward 方法）')
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig('figs/fig_E_dendrogram.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_E_dendrogram.png 已保存')

---
## 图6：轮廓系数图（不同 K 值对比）


In [ ]:
sil_scores = []
K_range2   = range(2, 9)
for k in K_range2:
    km    = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    labs  = km.fit_predict(X_km)
    sil_scores.append(silhouette_score(X_km, labs))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(K_range2, sil_scores, color=C['primary'], alpha=0.75)

best_k = list(K_range2)[np.argmax(sil_scores)]
ax.bar(best_k, max(sil_scores), color=C['highlight'], alpha=0.9,
       label=f'最优 K={best_k}（轮廓系数={max(sil_scores):.3f}）')

ax.set_xlabel('聚类数 K')
ax.set_ylabel('平均轮廓系数')
ax.set_title('不同 K 值的轮廓系数对比')
ax.set_xticks(list(K_range2))
ax.set_ylim(0, 1)
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig('figs/fig_E_silhouette.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_E_silhouette.png 已保存')

---
## 图7：孤立森林异常检测


In [ ]:
# 正常数据 + 人工注入异常点
np.random.seed(SEED)
X_normal  = np.random.multivariate_normal([0,0],[[1,0.5],[0.5,1]], 200)
X_outlier = np.random.uniform(-5, 5, (10, 2))  # 随机散布的异常点
X_all     = np.vstack([X_normal, X_outlier])

# 拟合孤立森林（contamination=5% 对应 10/210 ≈ 4.8%）
iforest = IsolationForest(n_estimators=200, contamination=0.05, random_state=SEED)
pred    = iforest.fit_predict(X_all)   # 1: 正常，-1: 异常
scores  = iforest.score_samples(X_all) # 分数越低越异常

fig, ax = plt.subplots(figsize=(6.5, 5.5))

# 绘制决策边界（异常分数等高线）
xx, yy = np.meshgrid(np.linspace(-6,6,150), np.linspace(-6,6,150))
Z = iforest.score_samples(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, levels=20, cmap='Blues', alpha=0.4)

# 正常点
mask_n = pred == 1
ax.scatter(X_all[mask_n,0], X_all[mask_n,1],
           color=C['primary'], alpha=0.6, s=25, label=f'正常点（{mask_n.sum()}个）')
# 异常点
mask_a = pred == -1
ax.scatter(X_all[mask_a,0], X_all[mask_a,1],
           color=C['highlight'], s=80, marker='*', zorder=5,
           label=f'检测异常点（{mask_a.sum()}个）', edgecolors='k', lw=0.5)

ax.set_xlim(-6,6); ax.set_ylim(-6,6)
ax.set_xlabel('特征 1'); ax.set_ylabel('特征 2')
ax.set_title('孤立森林异常检测结果\n（颜色深度代表异常分数，越深越可能是异常点）')
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig('figs/fig_E_isolation_forest.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_E_isolation_forest.png 已保存')

In [ ]:
print('\n' + '='*50)
print('Chapter E codes.ipynb 所有图形生成完成')
for f in sorted(os.listdir('figs')):
    if f.startswith('fig_E'):
        print(f'  {f}  ({os.path.getsize("figs/"+f)//1024} KB)')